# BLIP Fine-Tuning Experiments with MLflow



| Run | Learning rate | Train subset | Decoding evaluated |
|---|---|---|---|
| exp1_baseline | 5e-5 | 200 | greedy + beam |
| exp2_high_lr | 1e-4 | 200 | greedy + beam |
| exp3_large_subset | 5e-5 | 500 | greedy + beam |

In [4]:
import os
import shutil
import sys

IN_COLAB = "google.colab" in sys.modules
print("Running in Colab:", IN_COLAB)

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

    get_ipython().system("pip install -q mlflow evaluate rouge-score")

    DRIVE_DATASET = "/content/drive/MyDrive/flickr8k"
    LOCAL_DATASET = "/content/data/flickr8k"

    assert os.path.exists(DRIVE_DATASET)
    if not os.path.exists(LOCAL_DATASET):
        print("Copying dataset")
        shutil.copytree(DRIVE_DATASET, LOCAL_DATASET)
        print("Copy complete.")
    else:
        print("Dataset already copied - skipping.")


    if os.path.exists("/content/drive/MyDrive/mlflow.db") and not os.path.exists("/content/mlflow.db"):
        shutil.copy("/content/drive/MyDrive/mlflow.db", "/content/mlflow.db")
        print("Existing mlflow.db copied from Drive.")

    print("\nDataset folder contents:", os.listdir(LOCAL_DATASET))

Running in Colab: True
Mounted at /content/drive
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.7/49.7 kB 2.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.5/50.5 kB 5.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 81.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 55.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 44.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 264.7/264.7 kB 16.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 71.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 148.8/148.8 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

## 1. Imports

In [5]:
import json
import random
import time
from pathlib import Path

import evaluate
import mlflow
import nltk
import pandas as pd
import torch
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm
from transformers import BlipForConditionalGeneration, BlipProcessor

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)
if IN_COLAB and DEVICE.type != "cuda":
    print("WARNING: GPU not enabled! Runtime -> Change runtime type -> T4 GPU, then Run all again.")

Using device: cuda


## 2. Configuration

In [6]:
# ----------------------------- CONFIG -----------------------------
if IN_COLAB:
    DATA_ROOT = Path("/content/data/flickr8k")
    MLFLOW_TRACKING_URI = "sqlite:////content/mlflow.db"
    OUTPUT_DIR = Path("/content/outputs/day4_experiments")
else:
    DATA_ROOT = Path("../data/flickr8k")
    MLFLOW_TRACKING_URI = "sqlite:///mlflow.db"
    OUTPUT_DIR = Path("outputs/day4_experiments")

CAPTION_FILE = DATA_ROOT / "captions.txt"

# Handle both "Images" and "images" folder names automatically
IMAGE_DIR = DATA_ROOT / "Images"
if not IMAGE_DIR.exists():
    IMAGE_DIR = DATA_ROOT / "images"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MODEL_NAME = "Salesforce/blip-image-captioning-base"
RANDOM_SEED = 42
NUM_EPOCHS = 2
BATCH_SIZE = 8 if torch.cuda.is_available() else 2
EVAL_SAMPLE_SIZE = 150 if torch.cuda.is_available() else 60

EXPERIMENTS = [
    {"run_name": "exp1_baseline",     "learning_rate": 5e-5, "train_size": 200},
    {"run_name": "exp2_high_lr",      "learning_rate": 1e-4, "train_size": 200},
    {"run_name": "exp3_large_subset", "learning_rate": 5e-5, "train_size": 500},
]

MLFLOW_EXPERIMENT_NAME = "BLIP-Flickr8k-Captioning"
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_experiment(MLFLOW_EXPERIMENT_NAME)

assert CAPTION_FILE.exists(), f"captions.txt not found at {CAPTION_FILE}"
assert IMAGE_DIR.exists(), f"Images folder not found under {DATA_ROOT}"
print("Captions:", CAPTION_FILE)
print("Images:", IMAGE_DIR)
print("MLflow:", MLFLOW_TRACKING_URI, "| experiment:", MLFLOW_EXPERIMENT_NAME)
print("Batch size:", BATCH_SIZE, "| Eval images:", EVAL_SAMPLE_SIZE)

Captions: /content/data/flickr8k/captions.txt
Images: /content/data/flickr8k/Images
MLflow: sqlite:////content/mlflow.db | experiment: BLIP-Flickr8k-Captioning
Batch size: 8 | Eval images: 150


## 3. Data — Fixed Eval Subset + Disjoint Train Pool

Eval subset uses the same seed-42 sampling; training images come from the remaining pool (no leakage).

In [7]:
captions = pd.read_csv(CAPTION_FILE)
captions.columns = [str(c).strip().lower().replace(" ", "_") for c in captions.columns]

all_images = [img for img in captions["image"].drop_duplicates().tolist()
              if (IMAGE_DIR / img).exists()]
print("Usable images found:", len(all_images))

random.seed(RANDOM_SEED)
validation_images_300 = random.sample(all_images, 300)
eval_images = validation_images_300[:EVAL_SAMPLE_SIZE]

train_pool = [img for img in all_images if img not in set(validation_images_300)]
random.seed(RANDOM_SEED)
random.shuffle(train_pool)

eval_references = {img: captions.loc[captions["image"] == img, "caption"].astype(str).tolist()
                   for img in eval_images}

print(f"Eval subset: {len(eval_images)} | Train pool: {len(train_pool)}")

Usable images found: 576
Eval subset: 150 | Train pool: 276


## 4. Dataset, Model, Training Helpers

In [8]:
class Flickr8kCaptionDataset(Dataset):
    def __init__(self, image_names, captions_df, image_dir):
        self.image_names = image_names
        self.captions_df = captions_df
        self.image_dir = image_dir

    def __len__(self):
        return len(self.image_names)

    def __getitem__(self, index):
        image_name = self.image_names[index]
        with Image.open(self.image_dir / image_name) as image:
            rgb_image = image.convert("RGB")
        options = self.captions_df.loc[
            self.captions_df["image"] == image_name, "caption"
        ].astype(str).tolist()
        return rgb_image, random.choice(options)


def make_collate_fn(processor):
    def collate(batch):
        images, texts = zip(*batch)
        inputs = processor(images=list(images), text=list(texts),
                           return_tensors="pt", padding=True)
        labels = inputs["input_ids"].clone()
        labels[labels == processor.tokenizer.pad_token_id] = -100
        inputs["labels"] = labels
        return inputs
    return collate


def build_model():
    processor = BlipProcessor.from_pretrained(MODEL_NAME)
    model = BlipForConditionalGeneration.from_pretrained(MODEL_NAME).to(DEVICE)
    for p in model.vision_model.parameters():
        p.requires_grad = False   # freeze vision encoder - train text decoder only
    return processor, model


def train_model(model, processor, train_images, learning_rate, run_prefix=""):
    loader = DataLoader(Flickr8kCaptionDataset(train_images, captions, IMAGE_DIR),
                        batch_size=BATCH_SIZE, shuffle=True,
                        collate_fn=make_collate_fn(processor))
    optimizer = torch.optim.AdamW(
        [p for p in model.parameters() if p.requires_grad], lr=learning_rate)

    model.train()
    global_step = 0
    epoch_losses = []
    for epoch in range(1, NUM_EPOCHS + 1):
        running = 0.0
        progress = tqdm(loader, desc=f"{run_prefix}Epoch {epoch}/{NUM_EPOCHS}")
        for batch in progress:
            batch = {k: v.to(DEVICE) for k, v in batch.items()}
            loss = model(**batch).loss
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()
            global_step += 1
            running += loss.item()
            mlflow.log_metric("train_loss_step", loss.item(), step=global_step)
            progress.set_postfix(loss=f"{loss.item():.2f}")
        avg = running / len(loader)
        epoch_losses.append(avg)
        mlflow.log_metric("train_loss_epoch", avg, step=epoch)
        print(f"Epoch {epoch}: average training loss = {avg:.4f}")
    return epoch_losses

In [9]:
for resource in ["wordnet", "omw-1.4", "punkt", "punkt_tab"]:
    try:
        nltk.data.find(resource)
    except LookupError:
        nltk.download(resource, quiet=True)

bleu_metric = evaluate.load("bleu")
rouge_metric = evaluate.load("rouge")
meteor_metric = evaluate.load("meteor")

DECODING_STRATEGIES = {
    "greedy": {"max_new_tokens": 30, "num_beams": 1, "do_sample": False},
    "beam":   {"max_new_tokens": 30, "num_beams": 5, "early_stopping": True},
}


def evaluate_model(model, processor, strategy_name):
    model.eval()
    generation_config = DECODING_STRATEGIES[strategy_name]
    predictions, references = [], []
    for image_name in tqdm(eval_images, desc=f"Eval ({strategy_name})"):
        with Image.open(IMAGE_DIR / image_name) as image:
            rgb = image.convert("RGB")
        inputs = {k: v.to(DEVICE) for k, v in processor(images=rgb, return_tensors="pt").items()}
        with torch.inference_mode():
            out = model.generate(**inputs, **generation_config)
        predictions.append(processor.decode(out[0], skip_special_tokens=True).strip())
        references.append(eval_references[image_name])

    bleu = bleu_metric.compute(predictions=predictions, references=references)
    rouge = rouge_metric.compute(predictions=predictions, references=references)
    meteor = meteor_metric.compute(predictions=predictions, references=references)
    return {
        "BLEU": round(bleu["bleu"] * 100, 2),
        "ROUGE-1": round(float(rouge["rouge1"]) * 100, 2),
        "ROUGE-L": round(float(rouge["rougeL"]) * 100, 2),
        "METEOR": round(meteor["meteor"] * 100, 2),
    }, predictions

[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


## 5. Run All Experiments

In [10]:
all_results = []
start_time = time.time()

for experiment_config in EXPERIMENTS:
    run_name = experiment_config["run_name"]
    learning_rate = experiment_config["learning_rate"]
    train_size = experiment_config["train_size"]
    train_images = train_pool[:train_size]

    print("=" * 60)
    print(f"{run_name}: lr={learning_rate}, train_size={train_size}")
    print("=" * 60)

    processor, model = build_model()

    with mlflow.start_run(run_name=run_name) as run:
        mlflow.set_tag("stage", "day4_experiments")
        mlflow.set_tag("fine_tuned", "true")
        mlflow.set_tag("frozen_component", "vision_encoder")

        mlflow.log_param("model_name", MODEL_NAME)
        mlflow.log_param("learning_rate", learning_rate)
        mlflow.log_param("train_sample_size", train_size)
        mlflow.log_param("num_epochs", NUM_EPOCHS)
        mlflow.log_param("batch_size", BATCH_SIZE)
        mlflow.log_param("eval_sample_size", len(eval_images))
        mlflow.log_param("random_seed", RANDOM_SEED)
        mlflow.log_param("device", str(DEVICE))

        epoch_losses = train_model(model, processor, train_images,
                                   learning_rate, run_prefix=f"[{run_name}] ")

        row = {"run_name": run_name, "learning_rate": learning_rate,
               "train_size": train_size,
               "final_train_loss": round(epoch_losses[-1], 4)}

        for strategy_name in DECODING_STRATEGIES:
            metrics, predictions = evaluate_model(model, processor, strategy_name)
            for metric_name, value in metrics.items():
                mlflow.log_metric(f"{strategy_name}_{metric_name.replace('-', '_')}", value)
                row[f"{strategy_name}_{metric_name}"] = value
            pd.DataFrame({"image": eval_images, "caption": predictions}).to_csv(
                OUTPUT_DIR / f"{run_name}_{strategy_name}_captions.csv", index=False)

        all_results.append(row)
        print(f"Finished {run_name} - run_id: {run.info.run_id}")

    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print(f"\nAll experiments finished in {(time.time()-start_time)/60:.1f} min.")

exp1_baseline: lr=5e-05, train_size=200


preprocessor_config.json:   0%|          | 0.00/287 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/4.56k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/506 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

...

[exp3_large_subset] Epoch 2/2:   0%|          | 0/35 [00:00<?, ?it/s]

Epoch 2: average training loss = 2.2905


Eval (greedy):   0%|          | 0/150 [00:00<?, ?it/s]

Eval (beam):   0%|          | 0/150 [00:00<?, ?it/s]

Finished exp3_large_subset - run_id: 07defe96a9174f72b6bf2c2afa1c9e68

All experiments finished in 7.6 min.


## 6. Comparison and Best Configuration

In [11]:
comparison_df = pd.DataFrame(all_results)
comparison_df.to_csv(OUTPUT_DIR / "day4_run_comparison.csv", index=False)
print(comparison_df.to_string(index=False))

         run_name  learning_rate  train_size  final_train_loss  greedy_BLEU  greedy_ROUGE-1  greedy_ROUGE-L  greedy_METEOR  beam_BLEU  beam_ROUGE-1  beam_ROUGE-L  beam_METEOR
    exp1_baseline        0.00005         200            2.4891        21.78           52.36           48.71          45.09      23.59         53.45         50.67        46.55
     exp2_high_lr        0.00010         200            2.4488        18.89           50.60           47.71          45.24      23.76         53.59         50.87        49.42
exp3_large_subset        0.00005         500            2.2905        21.68           53.59           49.77          46.78      22.87         53.59         50.35        47.58


In [12]:
SELECTION_METRIC = "beam_METEOR"
best_row = comparison_df.loc[comparison_df[SELECTION_METRIC].idxmax()]

best_config = {
    "run_name": str(best_row["run_name"]),
    "learning_rate": float(best_row["learning_rate"]),
    "train_size": int(best_row["train_size"]),
    "decoding_strategy": "beam",
    "selection_metric": SELECTION_METRIC,
    "selection_value": float(best_row[SELECTION_METRIC]),
    "num_epochs": NUM_EPOCHS,
    "batch_size": BATCH_SIZE,
    "eval_sample_size": len(eval_images),
}
with open(OUTPUT_DIR / "best_config.json", "w") as f:
    json.dump(best_config, f, indent=2)

print("Best configuration (saved to best_config.json):")
print(json.dumps(best_config, indent=2))

Best configuration (saved to best_config.json):
{
  "run_name": "exp2_high_lr",
  "learning_rate": 0.0001,
  "train_size": 200,
  "decoding_strategy": "beam",
  "selection_metric": "beam_METEOR",
  "selection_value": 49.42,
  "num_epochs": 2,
  "batch_size": 8,
  "eval_sample_size": 150
}


##  Which Configuration Performed Best and Why
-Best configuration: exp3_large_subset (lr 5e-5, 500 images, beam)

-Learning rate: 5e-5 was more stable and scored higher; 1e-4 slightly hurt metrics

-Subset size: 500 clearly beat 200 (lowest loss 2.29, highest METEOR)

-Decoding: beam search beat greedy on every run (e.g., METEOR 45.2 → 49.4 for exp2), at ~5x slower generation

-Caveat: small-scale runs (2 epochs, 150-image eval) — conclusions are directional